# 🚀 Dynamic RAG Pipeline (Colab/Notebook Execution Guide)

This notebook sets up, tests, and patches the **master_code_scalable_robust.py** RAG pipeline.  
We explored two startup strategies (Plan A vs Plan B), then added a **final patch** for robust LLM handling and concise answers.

---

## ✅ Plan A: Inline (notebook-driven) server + test

- Monkey-patch memory limits, then import the module.
- Start `uvicorn` inside the notebook with logs.
- Use inline health-check + `/query` tests.
- Advantage: **self-contained**, all inside the notebook.
- Disadvantage: can be brittle in Colab (socket reuse, multiple processes).

**Code blocks in Plan A:**
1. Health-check/test helper functions.
2. Start Uvicorn with logs (subprocess).
3. Run `/health` and `/query`.
4. Kill stray servers at the end.

---

## ✅ Plan B: Script-first then notebook test

- First run `%run master_code_scalable_robust.py` to build the vector store and load the app code.
- Then in the notebook:
  - Kill any old `uvicorn`.
  - Start a new `uvicorn` process serving `master_code_scalable_robust:app`.
  - Run health and query tests via `requests`.

**Advantages:**  
- Clearer separation: pipeline logic is in the script, orchestration in notebook.  
- More stable in Colab runtime.  

---

## ✅ Final Patch: Robust prompt + model handling (≤140 chars)

We patched **`master_code_scalable_robust.py`** with two big improvements:

### 🔹 `get_llm()`
- **Tries HuggingFace Inference** if `HUGGINGFACEHUB_API_TOKEN` is set.
- If HF fails or no token: **falls back to local FLAN-T5 (google/flan-t5-base)**.
- Always logs which backend is used.

### 🔹 `retrieve_and_generate()`
- Uses **richer prompt template**:
  - "Answer ONLY using the context."
  - "If not in context, say so."
  - "Entire answer must be ≤ 140 characters."
- Adds `_enforce_char_limit()` helper:  
  - Strips whitespace.  
  - Cuts cleanly at last word.  
  - Adds ellipsis (`…`) if truncated.  
- Ensures both notebook calls and `/query` endpoint respect the ≤140 char rule.

---

## ✅ Test Procedure (after patch)

1. Kill any old server:
   ```python
   !pkill -f 'uvicorn master_code_scalable_robust:app' || true


2. Restart uvicorn with logs (force local fallback by unsetting HF token):

import os, sys, subprocess
env = os.environ.copy()
env.pop("HUGGINGFACEHUB_API_TOKEN", None)
subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "master_code_scalable_robust:app",
     "--host", "127.0.0.1", "--port", "8000", "--log-level", "info"],
    env=env
)


Test endpoints:

import requests
print(requests.get("http://127.0.0.1:8000/health").json())
print(requests.post("http://127.0.0.1:8000/query",
    json={"query": "List key WHO recommendations for infant feeding."}).json())

🔑 Key Takeaways

Plan A = inline server/test loop.

Plan B = run script then serve app via uvicorn (preferred in Colab).

Final patch makes the system:

Robust to missing/invalid HF token.

Always falls back to a local model.

Uses a concise, context-only prompt.

Guarantees ≤ 140 character answers.

This ensures both /query endpoint and notebook-level calls behave consistently and predictably.


---

Would you like me to also include a **diagram-style ASCII flowchart** (showing where

In [1]:
# # === 1) Install all dependencies (Colab-safe) ===
# !pip install --quiet --no-cache-dir faiss-cpu==1.8.0.post1 \
#   langchain langchain-community langchain-huggingface langchain-core langgraph \
#   fastapi uvicorn pydantic requests python-dotenv chromadb

# print("✅ Packages installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 158.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 134.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 170.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 185.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.2/153.2 kB 193.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.7/19.7 MB 167.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 216.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 165.6 MB

In [5]:
!pip install --quiet --no-cache-dir faiss-cpu==1.8.0.post1 \
  langchain langchain-community langchain-huggingface langchain-core langgraph \
  fastapi uvicorn pydantic requests python-dotenv chromadb \
  sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 249.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 223.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 182.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 161.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 194.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 157.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 215.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 176.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 215.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 206.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 223.8 MB/s eta 0:00:00


In [12]:
# Install local LLM deps
!pip install -q transformers accelerate


In [13]:
# === 2) Minimal environment setup ===
import os

# If you intend to call the HuggingFaceEndpoint LLM (google/gemma-2b-it), set your HF token here.
# You can create one at https://huggingface.co/settings/tokens
os.environ.setdefault("HUGGINGFACEHUB_API_TOKEN", "YOUR_HF_TOKEN_HERE")

# Set a polite user agent so remote sites aren't confused
os.environ.setdefault("USER_AGENT", "AllInOne-RAG/0.1 (contact: you@example.com)")

print("HUGGINGFACEHUB_API_TOKEN set:", bool(os.environ.get("HUGGINGFACEHUB_API_TOKEN")))
print("USER_AGENT:", os.environ.get("USER_AGENT"))


HUGGINGFACEHUB_API_TOKEN set: True
USER_AGENT: AllInOne-RAG/0.1 (contact: you@example.com)


In [14]:
%%writefile master_code_scalable_robust.py
# master_code_scalable_robust.py
"""
Unified and Scalable RAG Pipeline with a Persistent Vector Store.
Robust version: removes brittle global-flag checks, supports FAISS/Chroma fallback,
and adds a /health endpoint.
"""
import os
import logging
from typing import List, TypedDict, Dict, Any, Optional
from collections.abc import Iterable
from contextlib import asynccontextmanager

from dotenv import load_dotenv

# ==============================================================================
# SECTION 1: CONFIGURATION
# ==============================================================================
DOCUMENT_SOURCES = [
    "https://www.who.int/news-room/fact-sheets/detail/infant-and-young-child-feeding"
]
VECTOR_STORE_PATH = "faiss_vector_store"   # used for FAISS or as Chroma persist dir
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_REPO_ID = "google/gemma-2b-it"
LLM_TASK = "text-generation"
TOP_K_RESULTS = 3
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOXIC_KEYWORDS = {"kill", "hate", "suicide", "abuse", "stupid", "idiot", "die", "murder"}
PROFANITY_WORDS = {"damn", "shit", "fuck", "bitch", "bastard"}

# ==============================================================================
# SECTION 2: UTILITIES & LAZY LOADING
# ==============================================================================
_models: Dict[str, Any] = {}

def _load_model(key: str, loader):
    if key not in _models:
        _models[key] = loader()
    return _models[key]

def get_embedding_model():
    from langchain_huggingface import HuggingFaceEmbeddings
    return _load_model("embedding", lambda: HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL))

def get_llm():
    """
    Uses HuggingFaceEndpoint for generation.
    Requires HUGGINGFACEHUB_API_TOKEN in env.
    """
    from langchain_huggingface import HuggingFaceEndpoint
    api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
    if not api_token:
        raise ValueError("HUGGINGFACEHUB_API_TOKEN not set!")
    return _load_model(
        "llm",
        lambda: HuggingFaceEndpoint(
            repo_id=LLM_REPO_ID,
            task=LLM_TASK,
            temperature=0.1,
            max_new_tokens=512,
            huggingfacehub_api_token=api_token,
        ),
    )


# ==============================================================================
# SECTION 3: VECTOR STORE MANAGEMENT (FAISS -> Chroma fallback)
# ==============================================================================
# --- Vector store backend selection (FAISS preferred, fallback to Chroma) ---
_USE_FAISS = True
try:
    import faiss  # noqa: F401  # test import
except Exception:
    _USE_FAISS = False
    logging.warning("FAISS not available; falling back to Chroma vector store.")

if _USE_FAISS:
    from langchain_community.vectorstores import FAISS as VS
else:
    from langchain_community.vectorstores import Chroma as VS

from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def normalize_sources(sources: Any) -> List[str]:
    """
    Accepts str or iterable[str]; returns a concrete list[str].
    """
    if sources is None:
        return []
    if isinstance(sources, str):
        return [sources]
    if isinstance(sources, Iterable):
        out = [str(s).strip() for s in sources if s]
        return [s for s in out if s]
    raise TypeError("sources must be a str or an iterable of str")

class VectorStoreManager:
    def __init__(self, path: str = VECTOR_STORE_PATH):
        self.path = path
        self.embedding_model = get_embedding_model()
        self.vector_store: Optional[Any] = None

    def _get_loader(self, source: str):
        # Local PDFs
        if source.lower().endswith(".pdf") and (source.startswith("/") or "://" not in source):
            return PyPDFLoader(source)
        # HTTP/HTTPS (web pages or remote PDFs)
        if source.startswith("http://") or source.startswith("https://"):
            return WebBaseLoader([source])
        logging.warning(f"No loader available for source: {source}")
        return None

    def build_and_save(self, sources: Any) -> bool:
        logging.info("Building vector store...")
        sources = normalize_sources(sources)
        if not sources:
            logging.error("No sources provided. Vector store not built.")
            return False

        docs = []
        for src in sources:
            loader = self._get_loader(src)
            if not loader:
                continue
            try:
                loaded = loader.load()
                docs.extend(loaded)
                logging.info(f"Loaded {len(loaded)} docs from {src}")
            except Exception as e:
                logging.error(f"Failed to load source {src}: {e}", exc_info=True)

        if not docs:
            logging.error("No documents were loaded. Vector store not built.")
            return False

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
        )
        chunks = splitter.split_documents(docs)
        logging.info(f"Split {len(docs)} raw docs into {len(chunks)} chunks.")

        if _USE_FAISS:
            self.vector_store = VS.from_documents(chunks, self.embedding_model)
            self.vector_store.save_local(self.path)
        else:
            self.vector_store = VS.from_documents(
                chunks, self.embedding_model, persist_directory=self.path
            )
            self.vector_store.persist()

        logging.info(f"Vector store built and saved to {self.path}")
        return True

    def load(self) -> bool:
        """
        Idempotently load the vector index from disk into memory.
        Returns True on success.
        """
        try:
            if self.vector_store is not None:
                return True
            if not os.path.exists(self.path):
                logging.warning(f"Vector store path '{self.path}' does not exist.")
                return False

            if _USE_FAISS:
                from langchain_community.vectorstores import FAISS
                self.vector_store = FAISS.load_local(
                    self.path, self.embedding_model, allow_dangerous_deserialization=True
                )
            else:
                from langchain_community.vectorstores import Chroma
                self.vector_store = Chroma(
                    embedding_function=self.embedding_model,
                    persist_directory=self.path
                )

            logging.info("Vector store loaded from disk.")
            return True
        except Exception as e:
            logging.error(f"Error loading vector store: {e}", exc_info=True)
            self.vector_store = None
            return False

# ==============================================================================
# SECTION 4: PIPELINE NODES
# ==============================================================================
from langchain_core.prompts import PromptTemplate

vector_store_manager = VectorStoreManager()

def classify_query(text: str) -> Dict[str, Any]:
    lower = text.lower()
    flags = set()
    if any(w in lower for w in TOXIC_KEYWORDS):
        flags.add("toxic")
    if any(w in lower for w in PROFANITY_WORDS):
        flags.add("profanity")
    return {"is_safe": not bool(flags), "flags": sorted(flags)}

def retrieve_and_generate(query: str) -> Dict[str, Any]:
    # Ensure the store is available; attempt lazy load if needed
    if vector_store_manager.vector_store is None:
        if not vector_store_manager.load():
            return {"answer": "Knowledge base not loaded.", "sources": []}

    try:
        retriever = vector_store_manager.vector_store.as_retriever(
            search_kwargs={"k": TOP_K_RESULTS}
        )
        docs = retriever.invoke(query)
        if not docs:
            # Graceful fallback: answer briefly with no context
            return {
                "answer": "I couldn't retrieve relevant context for that question from the knowledge base.",
                "sources": [],
            }

        context = "\n---\n".join([doc.page_content for doc in docs])
        sources = sorted(list({doc.metadata.get("source", "Unknown") for doc in docs}))

        prompt = PromptTemplate.from_template(
            "You are a helpful assistant. Use the context to answer the question.\n"
            "If the answer isn't in the context, say so briefly.\n\n"
            "Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"
        )
        llm = get_llm()
        rag_chain = prompt | llm
        answer = rag_chain.invoke({"context": context, "question": query})
        answer_text = answer.strip() if isinstance(answer, str) else str(answer)
        return {"answer": answer_text, "sources": sources}
    except Exception as e:
        logging.error(f"Error during RAG generation: {e}", exc_info=True)
        raise

# ==============================================================================
# SECTION 5: LANGGRAPH WORKFLOW
# ==============================================================================
from langgraph.graph import StateGraph, END
class RAGState(TypedDict):
    query: str
    is_safe: bool
    final_answer: Dict[str, Any]

def classify_node(state: RAGState) -> Dict[str, Any]:
    out = classify_query(state["query"])
    return {"is_safe": out["is_safe"]}

def generate_node(state: RAGState) -> Dict[str, Any]:
    if not state["is_safe"]:
        return {"final_answer": {"answer": "Query flagged as unsafe.", "sources": []}}
    return {"final_answer": retrieve_and_generate(state["query"])}

def make_rag_app():
    builder = StateGraph(RAGState)
    builder.add_node("classify", classify_node)
    builder.add_node("generate", generate_node)
    builder.set_entry_point("classify")
    builder.add_edge("classify", "generate")
    return builder.compile()

rag_app = make_rag_app()

# ==============================================================================
# SECTION 6: FASTAPI APPLICATION
# ==============================================================================
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

@asynccontextmanager
async def lifespan(app: FastAPI):
    load_dotenv()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
    logging.info("--- Application Startup ---")
    # Attempt to load the vector store at startup; it's ok if it fails (lazy load later).
    vector_store_manager.load()
    yield
    logging.info("--- Application Shutdown ---")

app = FastAPI(title="Scalable RAG API (Robust)", version="3.3.0", lifespan=lifespan)

class QueryRequest(BaseModel):
    query: str

class QueryResponse(BaseModel):
    answer: str
    sources: List[str]

@app.get("/health")
async def health():
    """
    Simple health endpoint to check server & vector store status.
    """
    status = {
        "vector_store_loaded": vector_store_manager.vector_store is not None,
        "vector_store_path_exists": os.path.exists(VECTOR_STORE_PATH),
        "embedding_model": EMBEDDING_MODEL,
        "llm_repo": LLM_REPO_ID,
        "backend": "FAISS" if _USE_FAISS else "Chroma",
    }
    return status

@app.post("/query", response_model=QueryResponse)
async def query_rag(request: QueryRequest):
    try:
        result = await rag_app.ainvoke({"query": request.query})
        final = result.get("final_answer", {})
        if not final:
            # Fallback safety
            final = {"answer": "No answer generated.", "sources": []}
        return QueryResponse(**final)
    except Exception as e:
        logging.error(f"Error processing /query: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=str(e))

# ==============================================================================
# SECTION 7: ONE-TIME VECTOR STORE BUILDER
# ==============================================================================
if __name__ == "__main__":
    print("--- Running Vector Store Builder (Robust) ---")
    load_dotenv()
    ok = VectorStoreManager().build_and_save(DOCUMENT_SOURCES)
    if ok:
        print("✅ Vector store built successfully.")
    else:
        print("❌ Failed to build vector store. Check logs above.")
    print("\nTo run the API server, use:")
    print("uvicorn master_code_scalable_robust:app --reload")


Overwriting master_code_scalable_robust.py


In [15]:
# === 4) Build the vector store ===
%run master_code_scalable_robust.py


--- Running Vector Store Builder (Robust) ---
✅ Vector store built successfully.

To run the API server, use:
uvicorn master_code_scalable_robust:app --reload


##Path A — Direct non-HTTP test (fastest)

####  Use this if you just want to test retrieval + generation without the server.

*   This path does not require the server to be running.
*   The hot patch only affects this notebook process (not the server).



In [19]:
# PLAN A  as the rag was showing issue through hugging face thus going by the local llm
import master_code_scalable_robust as rag

# IMPORTANT: clear any previously cached LLM so the patch takes effect
rag._models.pop("llm", None)

# Local fallback: small, CPU-friendly model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_huggingface import HuggingFacePipeline

def _local_get_llm():
    local_model = "google/flan-t5-base"  # You can switch to 'flan-t5-large' if faster hardware
    tok = AutoTokenizer.from_pretrained(local_model)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(local_model)
    gen_pipe = pipeline("text2text-generation", model=mdl, tokenizer=tok, max_new_tokens=256)
    return rag._load_model("llm", lambda: HuggingFacePipeline(pipeline=gen_pipe))

# Monkeypatch in-memory
rag.get_llm = _local_get_llm

# (Optional) Recompile the graph; direct calls don't strictly need this, but it’s safe:
try:
    rag.rag_app = rag.make_rag_app()
    print("✅ Patched get_llm() to local FLAN-T5 and recompiled graph.")
except Exception as e:
    print("Graph recompilation warning (safe to ignore for direct calls):", e)


✅ Patched get_llm() to local FLAN-T5 and recompiled graph.


In [21]:
def _retrieve_and_generate_trimmed(query: str) -> dict:
    if rag.vector_store_manager.vector_store is None and not rag.vector_store_manager.load():
        return {"answer": "Knowledge base not loaded.", "sources": []}

    retriever = rag.vector_store_manager.vector_store.as_retriever(search_kwargs={"k": rag.TOP_K_RESULTS})
    docs = retriever.invoke(query)
    if not docs:
        return {"answer": "I couldn't retrieve relevant context for that question.", "sources": []}

    # Cap context size
    PER_DOC_CHARS = 800
    OVERALL_CHARS = 1800
    context = "\n---\n".join([d.page_content[:PER_DOC_CHARS] for d in docs])[:OVERALL_CHARS]

    sources = sorted(list({d.metadata.get("source", "Unknown") for d in docs}))

    from langchain_core.prompts import PromptTemplate
    prompt = PromptTemplate.from_template(
        "You are a helpful assistant. Use the context to answer the question.\n"
        "If the answer isn't in the context, say so briefly.\n\n"
        "Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"
    )
    llm = rag.get_llm()
    answer = (prompt | llm).invoke({"context": context, "question": query})
    return {"answer": str(answer).strip(), "sources": sources}

# Swap in trimmed version
rag.retrieve_and_generate = _retrieve_and_generate_trimmed
print("✅ retrieve_and_generate now trims context to avoid 512-token limit.")

✅ retrieve_and_generate now trims context to avoid 512-token limit.


In [22]:
# Make sure the vector store is available (idempotent)
rag.vector_store_manager.load()

res = rag.retrieve_and_generate("List key WHO recommendations for infant and young child feeding.")
print(res.get("answer", "")[:800], "...")
print("Sources:", res.get("sources"))


Device set to use cpu


the "Comprehensive implementation plan on maternal, infant and young child nutrition", which aims to protect, promote and support appropriate infant and young child feeding ...
Sources: ['https://www.who.int/news-room/fact-sheets/detail/infant-and-young-child-feeding']


##Path B — API test via /query (server)

Because Uvicorn runs in a separate process, it imports the module from disk and won’t see notebook monkeypatches. So for the API path, we write the patch into the file, then (re)start the server.

In [40]:
# --- Fix get_llm() in the file (robust, with local fallback) ---
from pathlib import Path
import re

p = Path("master_code_scalable_robust.py")
src = p.read_text(encoding="utf-8")

new_get_llm = r'''
def get_llm():
    """
    Robust get_llm:
    - Try Hugging Face Inference endpoint (if HUGGINGFACEHUB_API_TOKEN is set).
    - If that fails or token is missing, fall back to a local FLAN‑T5 pipeline.
    """
    import os, logging
    from langchain_huggingface import HuggingFacePipeline
    # 1) Remote (HF Inference)
    api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
    if api_token:
        try:
            from langchain_huggingface import HuggingFaceEndpoint
            return _load_model(
                "llm",
                lambda: HuggingFaceEndpoint(
                    repo_id=LLM_REPO_ID,
                    task=LLM_TASK,
                    temperature=0.1,
                    max_new_tokens=512,
                    huggingfacehub_api_token=api_token,
                    provider="hf-inference",  # avoid provider='auto' mapping issue
                ),
            )
        except Exception as e:
            logging.warning(f"HF endpoint failed, falling back to local model: {e}")

    # 2) Local fallback (CPU-friendly)
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
    local_model = os.getenv("LOCAL_LLM_ID", "google/flan-t5-base")
    tok = AutoTokenizer.from_pretrained(local_model)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(local_model)
    gen_pipe = pipeline("text2text-generation", model=mdl, tokenizer=tok, max_new_tokens=256)
    return _load_model("llm", lambda: HuggingFacePipeline(pipeline=gen_pipe))
'''

# Replace the entire get_llm() definition block.
# Strategy: find "def get_llm(" and replace up to the next SECTION divider (====) or end of file.
start = src.find("def get_llm(")
if start == -1:
    raise RuntimeError("Couldn't find def get_llm() in master_code_scalable_robust.py")

# Find the next SECTION divider after get_llm (or end of file)
section_div = "\n# ============================================================================== "
end = src.find(section_div, start)
if end == -1:
    end = len(src)

patched = src[:start] + new_get_llm + src[end:]
p.write_text(patched, encoding="utf-8")
print("✅ Patched get_llm() with robust fallback.")


✅ Patched get_llm() with robust fallback.


In [45]:
%%writefile master_code_scalable_robust.py
# master_code_scalable_robust.py
"""
Unified and Scalable RAG Pipeline with a Persistent Vector Store.
- FAISS→Chroma fallback
- HF Inference -> local FLAN-T5 fallback
- Trimmed context to fit small models
- FastAPI /health and /query
"""
import os
import logging
from typing import List, TypedDict, Dict, Any, Optional
from collections.abc import Iterable
from contextlib import asynccontextmanager

from dotenv import load_dotenv

# ==============================================================================
# SECTION 1: CONFIGURATION
# ==============================================================================
DOCUMENT_SOURCES = [
    "https://www.who.int/news-room/fact-sheets/detail/infant-and-young-child-feeding"
]
VECTOR_STORE_PATH = "faiss_vector_store"   # FAISS dir OR Chroma persist dir
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_REPO_ID = "google/gemma-2b-it"
LLM_TASK = "text-generation"
TOP_K_RESULTS = 3
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
TOXIC_KEYWORDS = {"kill", "hate", "suicide", "abuse", "stupid", "idiot", "die", "murder"}
PROFANITY_WORDS = {"damn", "shit", "fuck", "bitch", "bastard"}

# ==============================================================================
# SECTION 2: UTILITIES & LAZY LOADING
# ==============================================================================
_models: Dict[str, Any] = {}

def _load_model(key: str, loader):
    if key not in _models:
        _models[key] = loader()
    return _models[key]

def get_embedding_model():
    from langchain_huggingface import HuggingFaceEmbeddings
    return _load_model("embedding", lambda: HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL))

def get_llm():
    """
    Robust get_llm:
    - Try HF Inference (if HUGGINGFACEHUB_API_TOKEN is set).
    - Fallback to local FLAN-T5 (CPU-friendly).
    """
    import logging
    api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

    # 1) Remote (HF Inference)
    if api_token:
        try:
            from langchain_huggingface import HuggingFaceEndpoint
            return _load_model(
                "llm",
                lambda: HuggingFaceEndpoint(
                    repo_id=LLM_REPO_ID,
                    task=LLM_TASK,
                    temperature=0.1,
                    max_new_tokens=512,
                    huggingfacehub_api_token=api_token,
                    provider="hf-inference",  # avoid provider 'auto' mapping issue
                ),
            )
        except Exception as e:
            logging.warning(f"HF endpoint failed, falling back to local model: {e}")

    # 2) Local fallback
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline  # lazy import
    from langchain_huggingface import HuggingFacePipeline
    local_model = os.getenv("LOCAL_LLM_ID", "google/flan-t5-base")
    tok = AutoTokenizer.from_pretrained(local_model)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(local_model)
    gen_pipe = pipeline("text2text-generation", model=mdl, tokenizer=tok, max_new_tokens=256)
    return _load_model("llm", lambda: HuggingFacePipeline(pipeline=gen_pipe))

# ==============================================================================
# SECTION 3: VECTOR STORE MANAGEMENT (FAISS -> Chroma fallback)
# ==============================================================================
_USE_FAISS = True
try:
    import faiss  # noqa: F401  # probe
except Exception:
    _USE_FAISS = False
    logging.warning("FAISS not available; falling back to Chroma vector store.")

if _USE_FAISS:
    from langchain_community.vectorstores import FAISS as VS
else:
    from langchain_community.vectorstores import Chroma as VS

from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

def normalize_sources(sources: Any) -> List[str]:
    if sources is None:
        return []
    if isinstance(sources, str):
        return [sources]
    if isinstance(sources, Iterable):
        return [str(s).strip() for s in sources if s]
    raise TypeError("sources must be a str or an iterable of str")

class VectorStoreManager:
    def __init__(self, path: str = VECTOR_STORE_PATH):
        self.path = path
        self.embedding_model = get_embedding_model()
        self.vector_store: Optional[Any] = None

    def _get_loader(self, source: str):
        # Local PDFs
        if source.lower().endswith(".pdf") and (source.startswith("/") or "://" not in source):
            return PyPDFLoader(source)
        # HTTP/HTTPS (web pages or remote PDFs)
        if source.startswith("http://") or source.startswith("https://"):
            return WebBaseLoader([source])
        logging.warning(f"No loader available for source: {source}")
        return None

    def build_and_save(self, sources: Any) -> bool:
        logging.info("Building vector store...")
        sources = normalize_sources(sources)
        if not sources:
            logging.error("No sources provided. Vector store not built.")
            return False

        docs = []
        for src in sources:
            loader = self._get_loader(src)
            if not loader:
                continue
            try:
                loaded = loader.load()
                docs.extend(loaded)
                logging.info(f"Loaded {len(loaded)} docs from {src}")
            except Exception as e:
                logging.error(f"Failed to load source {src}: {e}", exc_info=True)

        if not docs:
            logging.error("No documents were loaded. Vector store not built.")
            return False

        splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
        chunks = splitter.split_documents(docs)
        logging.info(f"Split {len(docs)} raw docs into {len(chunks)} chunks.")

        if _USE_FAISS:
            self.vector_store = VS.from_documents(chunks, self.embedding_model)
            self.vector_store.save_local(self.path)
        else:
            self.vector_store = VS.from_documents(chunks, self.embedding_model, persist_directory=self.path)
            self.vector_store.persist()

        logging.info(f"Vector store built and saved to {self.path}")
        return True

    def load(self) -> bool:
        try:
            if self.vector_store is not None:
                return True
            if not os.path.exists(self.path):
                logging.warning(f"Vector store path '{self.path}' does not exist.")
                return False

            if _USE_FAISS:
                from langchain_community.vectorstores import FAISS
                self.vector_store = FAISS.load_local(
                    self.path, self.embedding_model, allow_dangerous_deserialization=True
                )
            else:
                from langchain_community.vectorstores import Chroma
                self.vector_store = Chroma(
                    embedding_function=self.embedding_model,
                    persist_directory=self.path
                )

            logging.info("Vector store loaded from disk.")
            return True
        except Exception as e:
            logging.error(f"Error loading vector store: {e}", exc_info=True)
            self.vector_store = None
            return False

# ==============================================================================
# SECTION 4: PIPELINE NODES (with trimmed context)
# ==============================================================================
from langchain_core.prompts import PromptTemplate

vector_store_manager = VectorStoreManager()

def classify_query(text: str) -> Dict[str, Any]:
    lower = text.lower()
    flags = set()
    if any(w in lower for w in TOXIC_KEYWORDS):
        flags.add("toxic")
    if any(w in lower for w in PROFANITY_WORDS):
        flags.add("profanity")
    return {"is_safe": not bool(flags), "flags": sorted(flags)}

def retrieve_and_generate(query: str) -> Dict[str, Any]:
    # Ensure the store is available; attempt lazy load if needed
    if vector_store_manager.vector_store is None:
        if not vector_store_manager.load():
            return {"answer": "Knowledge base not loaded.", "sources": []}

    try:
        retriever = vector_store_manager.vector_store.as_retriever(search_kwargs={"k": TOP_K_RESULTS})
        docs = retriever.invoke(query)
        if not docs:
            return {"answer": "I couldn't retrieve relevant context from the knowledge base.", "sources": []}

        # Trim context to avoid small-model limits
        PER_DOC_CHARS = 800
        OVERALL_CHARS = 1800
        context = "\n---\n".join([d.page_content[:PER_DOC_CHARS] for d in docs])[:OVERALL_CHARS]
        sources = sorted(list({d.metadata.get("source", "Unknown") for d in docs}))

        prompt = PromptTemplate.from_template(
            "You are a helpful assistant. Use the context to answer the question.\n"
            "If the answer isn't in the context, say so briefly.\n\n"
            "Context:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"
        )
        llm = get_llm()
        rag_chain = prompt | llm
        answer = rag_chain.invoke({"context": context, "question": query})
        answer_text = answer.strip() if isinstance(answer, str) else str(answer)
        return {"answer": answer_text, "sources": sources}
    except Exception as e:
        logging.error(f"Error during RAG generation: {e}", exc_info=True)
        raise

# ==============================================================================
# SECTION 5: LANGGRAPH WORKFLOW
# ==============================================================================
from langgraph.graph import StateGraph, END
class RAGState(TypedDict):
    query: str
    is_safe: bool
    final_answer: Dict[str, Any]

def classify_node(state: RAGState) -> Dict[str, Any]:
    out = classify_query(state["query"])
    return {"is_safe": out["is_safe"]}

def generate_node(state: RAGState) -> Dict[str, Any]:
    if not state["is_safe"]:
        return {"final_answer": {"answer": "Query flagged as unsafe.", "sources": []}}
    return {"final_answer": retrieve_and_generate(state["query"])}

def make_rag_app():
    builder = StateGraph(RAGState)
    builder.add_node("classify", classify_node)
    builder.add_node("generate", generate_node)
    builder.set_entry_point("classify")
    builder.add_edge("classify", "generate")
    return builder.compile()

rag_app = make_rag_app()

# ==============================================================================
# SECTION 6: FASTAPI APPLICATION
# ==============================================================================
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

@asynccontextmanager
async def lifespan(app: FastAPI):
    load_dotenv()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
    logging.info("--- Application Startup ---")
    vector_store_manager.load()
    yield
    logging.info("--- Application Shutdown ---")

app = FastAPI(title="Scalable RAG API (Robust)", version="3.3.1", lifespan=lifespan)

class QueryRequest(BaseModel):
    query: str

class QueryResponse(BaseModel):
    answer: str
    sources: List[str]

@app.get("/health")
async def health():
    status = {
        "vector_store_loaded": vector_store_manager.vector_store is not None,
        "vector_store_path_exists": os.path.exists(VECTOR_STORE_PATH),
        "embedding_model": EMBEDDING_MODEL,
        "llm_repo": LLM_REPO_ID,
        "backend": "FAISS" if _USE_FAISS else "Chroma",
    }
    return status

@app.post("/query", response_model=QueryResponse)
async def query_rag(request: QueryRequest):
    try:
        result = await rag_app.ainvoke({"query": request.query})
        final = result.get("final_answer", {}) or {"answer": "No answer generated.", "sources": []}
        return QueryResponse(**final)
    except Exception as e:
        logging.error(f"Error processing /query: {e}", exc_info=True)
        raise HTTPException(status_code=500, detail=str(e))

# ==============================================================================
# SECTION 7: ONE-TIME VECTOR STORE BUILDER
# ==============================================================================
if __name__ == "__main__":
    print("--- Running Vector Store Builder (Robust) ---")
    load_dotenv()
    ok = VectorStoreManager().build_and_save(DOCUMENT_SOURCES)
    print("✅ Vector store built successfully." if ok else "❌ Failed to build vector store.")
    print("Run: uvicorn master_code_scalable_robust:app --reload")


Overwriting master_code_scalable_robust.py


In [49]:
# 🚦 Robust start + port wait + health + query + diagnostics
import os, sys, time, socket, subprocess, requests, json, textwrap

def port_in_use(host="127.0.0.1", port=8000):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.2)
        return s.connect_ex((host, port)) == 0

# 0) Kill any old server (ignore errors)
subprocess.run("pkill -f 'uvicorn master_code_scalable_robust:app'", shell=True)

# 1) Import sanity (catch syntax/import issues early)
try:
    import importlib
    rag = importlib.import_module("master_code_scalable_robust")
    importlib.reload(rag)
    print("✅ Module import OK")
except Exception as e:
    raise RuntimeError(f"Module import failed: {e}")

# 2) Start server (force local LLM by removing HF token; add unbuffered logs)
env = os.environ.copy()
env.pop("HUGGINGFACEHUB_API_TOKEN", None)  # force local FLAN-T5 fallback
env["PYTHONUNBUFFERED"] = "1"

cmd = [
    sys.executable, "-m", "uvicorn",
    "master_code_scalable_robust:app",
    "--host", "127.0.0.1",
    "--port", "8000",
    "--log-level", "info",
]
server = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env
)
print("🚀 Starting Uvicorn…")

# 3) Read logs until startup complete or crash
started = False
lines = []
t0 = time.time()
while time.time() - t0 < 30:
    line = server.stdout.readline()
    if line:
        lines.append(line.rstrip())
        print(line, end="")
        if "Application startup complete" in line:
            started = True
            break
    elif server.poll() is not None:
        break

# 4) If started, wait for port to open
if started:
    for i in range(30):  # up to ~6s
        if port_in_use():
            break
        time.sleep(0.2)
    print("Port open?:", port_in_use())
else:
    print("\n❌ Did not see 'Application startup complete'.")

print("Process running?:", server.poll() is None)

# If process died, show last 80 log lines and abort
if server.poll() is not None:
    print("\n🔎 Server exited. Return code:", server.returncode)
    print("\n--- Last logs ---")
    print("\n".join(lines[-80:]) or "(no logs)")
else:
    # 5) Health check with retry
    ok = False
    for i in range(10):
        try:
            r = requests.get("http://127.0.0.1:8000/health", timeout=3)
            print("\n/health:", r.status_code)
            print(json.dumps(r.json(), indent=2))
            ok = r.ok
            break
        except Exception as e:
            print(f"Retry health {i+1}: {e}")
            time.sleep(0.5)

    # 6) Query test (only if health OK)
    if ok:
        payload = {"query": "List key WHO recommendations for infant and young child feeding."}
        r = requests.post("http://127.0.0.1:8000/query", json=payload, timeout=120)
        print("\n/query:", r.status_code)
        if "application/json" in r.headers.get("content-type", ""):
            print(json.dumps(r.json(), indent=2))
        else:
            print(r.text)


✅ Module import OK
🚀 Starting Uvicorn…
2025-08-16 16:56:53.224545: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755363413.263610   29111 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755363413.275575   29111 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755363413.304444   29111 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755363413.304607   29111 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755363413.304633   29111 computatio

In [71]:
try:
    server.terminate()
    server.wait(timeout=10)
    print("🛑 Server stopped.")
except Exception as e:
    print("Couldn't stop server:", e)


🛑 Server stopped.


In [67]:
import importlib, textwrap

# Reload the module cleanly
import master_code_scalable_robust as rag
importlib.reload(rag)

# ---- PATCH retrieve_and_generate so both notebook & server API use it ----
def retrieve_and_generate(query: str, k: int = 5):
    """
    Retrieve relevant docs from FAISS and generate an answer
    using the configured LLM with a richer prompt.
    """
    rag.vector_store_manager.load()

    retriever = rag.vector_store_manager.as_retriever(k=k)
    docs = retriever.get_relevant_documents(query)
    context = "\n\n".join(d.page_content for d in docs)

    llm = rag.get_llm()

    from langchain.prompts import PromptTemplate
    template = textwrap.dedent("""
        You are a medical/parenting assistant.
        Use the context below to answer the user's question.

        Context:
        {context}

        Question:
        {question}

        Guidelines:
        - Answer in clear, concise, factual style
        - Highlight WHO or government recommendations if relevant.
        - If unsure, say "I don't know" rather than hallucinating.


        Answer:
    """)
    prompt = PromptTemplate.from_template(template)
    rag_chain = prompt | llm

    answer = rag_chain.invoke({"context": context, "question": query})

    return {
        "answer": answer.strip(),
        "sources": [d.metadata.get("source", "") for d in docs if "source" in d.metadata]
    }

# Monkey-patch into the module so FastAPI uses it too
rag.retrieve_and_generate = retrieve_and_generate

print("✅ Patched retrieve_and_generate in module and API")


✅ Patched retrieve_and_generate in module and API


In [68]:
try:
    server.terminate()
    server.wait(timeout=10)
    print("🛑 Old server stopped")
except Exception as e:
    print("Couldn't stop server:", e)


🛑 Old server stopped


In [69]:
# 🔁 Restart server forcing local fallback (no HF token)
import os, sys, time, subprocess, requests, json

# Stop old server
subprocess.run("pkill -f 'uvicorn master_code_scalable_robust:app'", shell=True)

# Launch without HUGGINGFACEHUB_API_TOKEN so get_llm() uses local FLAN-T5
env = os.environ.copy()
env.pop("HUGGINGFACEHUB_API_TOKEN", None)
env["PYTHONUNBUFFERED"] = "1"

cmd = [sys.executable, "-m", "uvicorn", "master_code_scalable_robust:app",
       "--host", "127.0.0.1", "--port", "8000", "--log-level", "info"]
server = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, env=env)

# Wait for startup
started = False
t0 = time.time()
while time.time() - t0 < 30:
    line = server.stdout.readline()
    if not line:
        if server.poll() is not None:
            break
        time.sleep(0.05); continue
    print(line, end="")
    if "Application startup complete" in line:
        started = True
        break

# Health + query
if started:
    r = requests.get("http://127.0.0.1:8000/health", timeout=10)
    print("\n/health:", r.status_code, r.text)
    payload = {"query": "List key WHO recommendations for infant and young child feeding."}
    r = requests.post("http://127.0.0.1:8000/query", json=payload, timeout=120)
    print("\n/query:", r.status_code)
    print(r.json() if "application/json" in r.headers.get("content-type","") else r.text)
else:
    print("❌ Server did not start.")


2025-08-16 17:53:55.174782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755366835.214607   42824 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755366835.226622   42824 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755366835.254928   42824 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755366835.254984   42824 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755366835.254987   42824 computation_placer.cc:177] computation placer alr

In [70]:
import requests

print("/health:", requests.get("http://127.0.0.1:8000/health").json())
print("/query:", requests.post("http://127.0.0.1:8000/query",
                               json={"query": "In 20 words summarize HIV and infant feeding."}).json())


/health: {'vector_store_loaded': True, 'vector_store_path_exists': True, 'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'llm_repo': 'google/gemma-2b-it', 'backend': 'FAISS'}
/query: {'answer': 'HIV and infant feedingBreastfeeding, and especially early and exclusive breastfeeding, is one of the most significant ways to improve infant survival rates.While HIV can pass from a mother to her child during pregnancy, labour or delivery, and also through breast-milk, the evidence on HIV and infant feeding shows that giving antiretroviral treatment (ART) to mothers living with HIV significantly reduces the risk of transmission through breastfeeding and also improves her health.WHO now recommends that all people living with HIV, including pregnant women and lactating mothers living with HIV, take ART for life from when they first learn their infection status.Mothers living in settings where morbidity and mortality due to diarrhoea, pneumonia and malnutrition ar

##Option B (permanent in code): force always-local LLM

##If you don’t want to worry about env tokens at all then replace get_llm() in the file to always use FLAN-T5:

In [64]:


# 🛠️ Permanently force local model in master_code_scalable_robust.py
from pathlib import Path

p = Path("master_code_scalable_robust.py")
src = p.read_text(encoding="utf-8")

new_get_llm = '''
def get_llm():
    """Always use local FLAN-T5 as the LLM."""
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
    from langchain_huggingface import HuggingFacePipeline
    local_model = os.getenv("LOCAL_LLM_ID", "google/flan-t5-base")
    tok = AutoTokenizer.from_pretrained(local_model)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(local_model)
    gen_pipe = pipeline("text2text-generation", model=mdl, tokenizer=tok, max_new_tokens=256)
    return _load_model("llm", lambda: HuggingFacePipeline(pipeline=gen_pipe))
'''

start = src.find("def get_llm(")
if start == -1:
    raise RuntimeError("Couldn't find def get_llm() in file.")
# Replace get_llm block up to next SECTION divider or EOF
divider = "\n# =============================================================================="
end = src.find(divider, start)
if end == -1: end = len(src)
patched = src[:start] + new_get_llm + src[end:]
p.write_text(patched, encoding="utf-8")
print("✅ get_llm() now always uses local FLAN-T5.")

# Restart server normally (no special env needed)
import subprocess, sys, time, requests, json
subprocess.run("pkill -f 'uvicorn master_code_scalable_robust:app'", shell=True)
server = subprocess.Popen([sys.executable, "-m", "uvicorn", "master_code_scalable_robust:app",
                           "--host", "127.0.0.1", "--port", "8000", "--log-level", "info"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1)
t0 = time.time(); started=False
while time.time()-t0<30:
    line = server.stdout.readline()
    if not line:
        if server.poll() is not None: break
        time.sleep(0.05); continue
    print(line, end="")
    if "Application startup complete" in line: started=True; break
if started:
    print("/health:", requests.get("http://127.0.0.1:8000/health").json())
    print("/query:", requests.post("http://127.0.0.1:8000/query",
                                   json={"query":"List key takeaways for HIV and infant feeding."}).json())
else:
    print("❌ Server did not start.")


✅ get_llm() now always uses local FLAN-T5.
2025-08-16 17:47:24.974449: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755366445.007989   41251 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755366445.020114   41251 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755366445.049111   41251 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755366445.049291   41251 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755366445.049323   41251 comput

In [65]:
import requests

print("/health:", requests.get("http://127.0.0.1:8000/health").json())
print("/query:", requests.post("http://127.0.0.1:8000/query",
                               json={"query": "Summarize HIV and infant feeding in 3 bullet points, and each bullet point should not exceed 10 words."}).json())


/health: {'vector_store_loaded': True, 'vector_store_path_exists': True, 'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 'llm_repo': 'google/gemma-2b-it', 'backend': 'FAISS'}
/query: {'answer': 'HIV and infant feedingBreastfeeding, and especially early and exclusive breastfeeding, is one of the most significant ways to improve infant survival rates.While HIV can pass from a mother to her child during pregnancy, labour or delivery, and also through breast-milk, the evidence on HIV and infant feeding shows that giving antiretroviral treatment (ART) to mothers living with HIV significantly reduces the risk of transmission through breastfeeding and also improves her health.WHO now recommends that all people living with HIV, including pregnant women and lactating mothers living with HIV, take ART for life from when they first learn their infection status.Mothers living in settings where morbidity and mortality due to diarrhoea, pneumonia and malnutrition ar

In [74]:
try:
    server.terminate()
    server.wait(timeout=10)
    print("🛑 Old server stopped")
except Exception as e:
    print("Couldn't stop server:", e)


🛑 Old server stopped


In [72]:
# 🛠️ Combined patch: update get_llm() + retrieve_and_generate() in master_code_scalable_robust.py
from pathlib import Path

path = Path("master_code_scalable_robust.py")
src  = path.read_text(encoding="utf-8")

GET_LLM_NEW = r'''
def get_llm():
    """
    Robust get_llm:
    - Try HF Inference (if HUGGINGFACEHUB_API_TOKEN is set).
    - Fallback to local FLAN-T5 (CPU-friendly).
    """
    import os, logging
    api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

    # 1) Remote (HF Inference)
    if api_token:
        try:
            from langchain_huggingface import HuggingFaceEndpoint
            return _load_model(
                "llm",
                lambda: HuggingFaceEndpoint(
                    repo_id=LLM_REPO_ID,
                    task=LLM_TASK,
                    temperature=0.1,
                    max_new_tokens=512,
                    huggingfacehub_api_token=api_token,
                    provider="hf-inference",  # avoid provider='auto' mapping issue
                ),
            )
        except Exception as e:
            logging.warning(f"HF endpoint failed, falling back to local model: {e}")

    # 2) Local fallback
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline  # lazy import
    from langchain_huggingface import HuggingFacePipeline
    local_model = os.getenv("LOCAL_LLM_ID", "google/flan-t5-base")
    tok = AutoTokenizer.from_pretrained(local_model)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(local_model)
    gen_pipe = pipeline("text2text-generation", model=mdl, tokenizer=tok, max_new_tokens=256)
    return _load_model("llm", lambda: HuggingFacePipeline(pipeline=gen_pipe))
'''

RAG_FUNC_NEW = r'''
def _enforce_char_limit(text: str, limit: int = 140) -> str:
    # Collapse whitespace, strip, then truncate without cutting the last word.
    s = " ".join((text or "").split()).strip()
    if len(s) <= limit:
        return s
    cut = s[: max(0, limit - 1)]
    if " " in cut:
        cut = cut[: cut.rfind(" ")].rstrip()
    return cut + "…"

def retrieve_and_generate(query: str) -> Dict[str, Any]:
    # Ensure the store is available; attempt lazy load if needed
    if vector_store_manager.vector_store is None:
        if not vector_store_manager.load():
            return {"answer": "Knowledge base not loaded.", "sources": []}

    try:
        retriever = vector_store_manager.vector_store.as_retriever(search_kwargs={"k": TOP_K_RESULTS})
        docs = retriever.invoke(query)
        if not docs:
            return {"answer": "No relevant context in the knowledge base.", "sources": []}

        # Trim context for small models
        PER_DOC_CHARS = 900
        OVERALL_CHARS = 2200
        context = "\\n---\\n".join([d.page_content[:PER_DOC_CHARS] for d in docs])[:OVERALL_CHARS]
        sources = sorted({d.metadata.get("source", "Unknown") for d in docs})

        from langchain_core.prompts import PromptTemplate
        prompt = PromptTemplate.from_template(
            "You are a concise, factual assistant. Answer ONLY using the context.\n"
            "Your ENTIRE answer must be <= 140 characters.\n"
            "If the answer is not in the context, say so briefly.\n\n"
            "Context:\\n{context}\\n\\nQuestion:\\n{question}\\n\\nAnswer (<=140 chars):"
        )
        llm = get_llm()
        rag_chain = prompt | llm
        raw = rag_chain.invoke({"context": context, "question": query})
        answer_text = raw.strip() if isinstance(raw, str) else str(raw)
        answer_text = _enforce_char_limit(answer_text, 140)
        return {"answer": answer_text, "sources": list(sources)}
    except Exception as e:
        logging.error(f"Error during RAG generation: {e}", exc_info=True)
        raise
'''

def replace_block(src_text: str, func_name: str, new_block: str) -> str:
    start = src_text.find(f"def {func_name}(")
    if start == -1:
        raise RuntimeError(f"Couldn't find def {func_name}() in the file.")
    divider = "\n# =============================================================================="
    end = src_text.find(divider, start)
    if end == -1:
        end = len(src_text)
    return src_text[:start] + new_block + src_text[end:]

# Replace both blocks
src = replace_block(src, "get_llm", GET_LLM_NEW)
src = replace_block(src, "retrieve_and_generate", RAG_FUNC_NEW)

path.write_text(src, encoding="utf-8")
print("✅ Patched get_llm() and retrieve_and_generate() (<=140 chars, richer prompt, HF→local fallback).")


✅ Patched get_llm() and retrieve_and_generate() (<=140 chars, richer prompt, HF→local fallback).


In [73]:
# 🔁 Restart server and test
import os, sys, time, subprocess, requests, json

# Stop old server
subprocess.run("pkill -f 'uvicorn master_code_scalable_robust:app'", shell=True)

# Launch WITHOUT HF token so we definitely use local fallback
env = os.environ.copy()
env.pop("HUGGINGFACEHUB_API_TOKEN", None)
env["PYTHONUNBUFFERED"] = "1"

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "master_code_scalable_robust:app",
     "--host", "127.0.0.1", "--port", "8000", "--log-level", "info"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env
)

# Wait for "Application startup complete"
started = False
t0 = time.time()
while time.time() - t0 < 30:
    line = server.stdout.readline()
    if not line:
        if server.poll() is not None: break
        time.sleep(0.05); continue
    print(line, end="")
    if "Application startup complete" in line:
        started = True
        break

# Health + query
if started:
    print("/health:", requests.get("http://127.0.0.1:8000/health", timeout=10).json())
    r = requests.post("http://127.0.0.1:8000/query",
                      json={"query":"List key WHO recommendations for infant and young child feeding."},
                      timeout=120)
    print("/query:", r.status_code)
    print(r.json())
else:
    print("❌ Server failed to start; check logs above.")


2025-08-16 18:04:15.245582: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755367455.269850   45309 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755367455.277111   45309 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1755367455.296170   45309 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755367455.296209   45309 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1755367455.296213   45309 computation_placer.cc:177] computation placer alr